In [1]:
# Step 1: Configure the MLflow server and credentials used to track the analysis. Do not commit real passwords.

MLFLOW_SERVER = "http://10.0.100.20:5000"
MLFLOW_USERNAME = "csc-4933-001-user01"
MLFLOW_PASSWORD = "dG7hrCzKxFyJfP9v"

# 1. Objective and Hypothesis 2

## Project Objective

The objective of this project is to investigate the environmental impact of replacing 10,000 gasoline-powered vehicles with electric vehicles (EVs) in a mid-sized city. The project examines electricity demand, grid-related carbon emissions, weather patterns, and the potential reduction in emissions from gasoline-powered transportation.

Using data from 2022 and 2023, the analysis compares the initial EV adoption period in 2022 with the first full year of EV operation in 2023. The goal is to understand how EV charging and environmental conditions relate to electricity demand and estimated grid emissions.

## Hypothesis 2 — Cold Weather and Grid Emissions

### Original Hypothesis

If outdoor temperatures decrease, then hourly electricity demand and estimated grid CO₂ emissions will increase because colder conditions can increase electricity use for heating and change the electricity generation mix.

### Reason for Revision

The original hypothesis was designed around hourly weather and electricity grid observations. However, the datasets being used for this analysis contain daily averages for both weather and electricity grid measurements.

To ensure that the analysis uses consistent time intervals, Hypothesis 2 was revised from hourly observations to daily observations. This allows daily average temperatures to be compared with daily average electricity demand and daily estimated grid CO₂ emissions.

This revision also means the analysis will focus on broader daily patterns rather than changes occurring within individual hours. As a result, the project will not make conclusions about specific hourly charging or electricity demand patterns.

### Corrected Hypothesis (M2)

If daily average outdoor temperatures decrease, then daily average electricity demand and daily estimated grid CO₂ emissions will increase because colder conditions can increase electricity use for heating and change the electricity generation mix.

### Predictions

The revised hypothesis will be evaluated using two separate predictions:

1. **Electricity Demand:** Days in the coldest daily temperature quartile will have higher average electricity demand than days in the warmest daily temperature quartile.

2. **Grid CO₂ Emissions:** Days in the coldest daily temperature quartile will have higher estimated grid CO₂ emissions than days in the warmest daily temperature quartile.

These predictions will be evaluated independently because increased electricity demand does not necessarily result in increased carbon emissions. The emissions impact also depends on the types of electricity generation sources being used.

## Analysis Approach

The analysis will use daily weather and electricity grid data from 2022 and 2023.

- **Experiments:** Compare electricity demand and estimated grid emissions between the coldest and warmest daily temperature groups.
- **Models:** Develop and compare candidate models using calendar features and daily average temperature to predict electricity demand and estimated grid emissions.
- **Validation:** Evaluate model performance using chronological training, validation, and testing periods to determine how well the models perform on unseen dates.

The analysis will focus on identifying relationships and predictive patterns between temperature, electricity demand, and estimated grid emissions. These relationships will not automatically be interpreted as proof that temperature directly caused changes in electricity demand or emissions.

## Important Limitations

The electricity grid data represents the broader MCPA balancing authority rather than only the 10,000 EVs or the city itself. Therefore, the analysis cannot attribute all observed changes in electricity demand or grid emissions directly to EV adoption.

Additionally, daily averages can hide differences between daytime and nighttime electricity use. Conclusions will therefore be limited to daily patterns rather than specific hourly behaviors.

# 2. Load and Inspect the Daily Datasets

This section loads the prepared daily dataset containing both electricity grid and weather measurements, along with the fuel emissions factor dataset.

The grid and weather data were previously cleaned, aggregated from hourly observations into daily measurements, and combined into one CSV file.

The combined dataset provides daily average outdoor temperatures, daily average electricity demand, and electricity generation measurements. The fuel emissions factor dataset provides the CO₂ emissions associated with each electricity generation source.

The purpose of this section is to verify that the prepared datasets are ready for the Hypothesis 2 experiments and modeling process.

## 2.1 Identify the Required Datasets

Two prepared datasets will be used for Hypothesis 2:

- **Combined daily averages dataset:** Contains daily average electricity demand, electricity generation by source, outdoor temperature, weather measurements, and the number of hours recorded for each day.
- **Fuel emissions factors:** Contains the CO₂ emissions factor, measured in kilograms of CO₂ per MWh, for each electricity generation source.

The combined daily dataset will provide the observations used in the experiments and models. The fuel emissions factors will be used to calculate estimated daily grid CO₂ emissions.

## 2.2 Import Libraries and Load the Datasets

The required Python libraries will be imported, and the prepared daily dataset and fuel emissions factor dataset will be loaded into separate DataFrames.

AI usage: help with matplot and pathlib syntax and functions, ex: .shape()

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

# Load the combined daily dataset
daily_data = pd.read_csv("combined_daily_averages.csv")

# Load the complete fuel emissions factor dataset
fuel_factors_clean = pd.read_csv("fuel_factors_clean.csv")

# Convert the date column to datetime format
daily_data["date"] = pd.to_datetime(daily_data["date"])

# Display dataset dimensions
print("Daily dataset:", daily_data.shape)
print("Fuel emissions factors:", fuel_factors_clean.shape)

display(daily_data.head())
display(fuel_factors_clean)

FileNotFoundError: [Errno 2] No such file or directory: 'combined_daily_averages.csv'

## 2.3 Inspect the Prepared Datasets

Both datasets will be inspected to verify their dimensions, column names, date coverage, missing values, and duplicate records.

The daily dataset will also be checked to confirm that the number of recorded hours is consistent with the daily aggregation process.

In [6]:
# Inspect the combined daily dataset
print("COMBINED DAILY DATASET")
print("Shape:", daily_data.shape)
print("Columns:", daily_data.columns.tolist())

print("\nDate range:")
print(daily_data["date"].min(), "to", daily_data["date"].max())

print("\nMissing values:")
display(daily_data.isnull().sum())

print("\nDuplicate dates:")
print(daily_data.duplicated(subset=["date"]).sum())

print("\nHours recorded per day:")
display(daily_data["hours_recorded"].value_counts().sort_index())

# Inspect the fuel emissions factor dataset
print("\nFUEL EMISSIONS FACTORS")
print("Shape:", fuel_factors_clean.shape)
print("Columns:", fuel_factors_clean.columns.tolist())

print("\nMissing values:")
display(fuel_factors_clean.isnull().sum())

print("\nDuplicate fuel entries:")
print(fuel_factors_clean["fuel"].duplicated().sum())

display(fuel_factors_clean)

COMBINED DAILY DATASET
Shape: (730, 16)
Columns: ['date', 'demand_mwh', 'demand_forecast_mwh', 'net_generation', 'coal', 'natural_gas', 'nuclear', 'wind', 'solar', 'hydro', 'other', 'temp_f', 'heating_degree_hours', 'wind_speed_mph', 'relative_humidity', 'hours_recorded']

Date range:
2022-01-01 00:00:00 to 2023-12-31 00:00:00

Missing values:


date                    0
demand_mwh              0
demand_forecast_mwh     0
net_generation          0
coal                    0
natural_gas             0
nuclear                 0
wind                    0
solar                   0
hydro                   0
other                   0
temp_f                  0
heating_degree_hours    0
wind_speed_mph          0
relative_humidity       0
hours_recorded          0
dtype: int64


Duplicate dates:
0

Hours recorded per day:


hours_recorded
23      2
24    726
25      2
Name: count, dtype: int64


FUEL EMISSIONS FACTORS
Shape: (8, 3)
Columns: ['fuel', 'kg_co2_per_mwh', 'note']

Missing values:


fuel              0
kg_co2_per_mwh    0
note              0
dtype: int64


Duplicate fuel entries:
0


,fuel,kg_co2_per_mwh,note
0,coal,1010.0,fleet average of the BA's coal units
1,natural_gas,430.0,fleet average of the BA's gas units (combined ...
2,nuclear,0.0,no stack emissions
3,wind,0.0,no stack emissions
4,solar,0.0,no stack emissions
5,hydro,0.0,no stack emissions
6,other,0.0,"landfill gas and biomass; biogenic, counted as..."
7,imports,520.0,average of the neighbouring balancing authorit...


## 3.1 Identify and Prepare the Required Columns

The combined daily dataset contains the electricity grid and weather measurements needed to evaluate Hypothesis 2.

The primary variables used in this analysis are:

- `date`: Date of the daily observation.
- `temp_f`: Daily average outdoor temperature in degrees Fahrenheit.
- `demand_mwh`: Daily average of hourly electricity consumption, measured in MWh.
- `net_generation`: Reported electricity generation across the available generation sources.
- `coal`, `natural_gas`, `nuclear`, `wind`, `solar`, `hydro`, and `other`: Electricity generation measurements by source.
- `hours_recorded`: Number of hourly observations recorded for each day.

The fuel generation variables will be used to estimate daily grid CO₂ emissions. Calendar variables will also be created to support the candidate prediction models.

In [7]:
# Identify the required columns
required_columns = [
    "date",
    "temp_f",
    "demand_mwh",
    "net_generation",
    "coal",
    "natural_gas",
    "nuclear",
    "wind",
    "solar",
    "hydro",
    "other",
    "hours_recorded"
]

# Check for missing columns
missing_columns = [
    column for column in required_columns
    if column not in daily_data.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Display the required columns
display(daily_data[required_columns].head())

print("All required columns are available.")

,date,temp_f,demand_mwh,net_generation,coal,natural_gas,nuclear,wind,solar,hydro,other,hours_recorded
0,2022-01-01,43.691667,743.625000,743.625000,302.708333,93.333333,210.083333,65.291667,10.458333,51.791667,9.958333,24
1,2022-01-02,43.258333,741.583333,741.625000,333.833333,94.583333,209.708333,29.083333,12.083333,52.333333,10.000000,24
2,2022-01-03,38.054167,863.083333,862.958333,396.333333,165.291667,210.208333,17.416667,10.625000,53.166667,9.916667,24
3,2022-01-04,42.395833,823.166667,823.416667,342.625000,108.000000,210.041667,88.708333,11.916667,52.000000,10.125000,24
4,2022-01-05,44.670833,791.458333,791.500000,351.833333,103.208333,210.041667,51.583333,12.250000,52.625000,9.958333,24


All required columns are available.


## 3.2 Calculate Daily Estimated Grid CO₂ Emissions

Daily estimated grid CO₂ emissions will be calculated using electricity generation by fuel source and the corresponding emissions factor.

The emissions factor represents the estimated kilograms of CO₂ produced for each MWh of electricity generated by a particular source.

The calculation includes all seven generation categories available in the combined daily dataset, including sources with zero operational CO₂ emissions.

The generation variables are treated as daily average hourly generation measurements. Under this assumption, each generation value is multiplied by the number of hours recorded for that day to estimate total daily generation in MWh.

The daily generation amount is then multiplied by its corresponding emissions factor. Emissions from all seven sources are added together to calculate estimated daily grid CO₂ emissions.

**Daily Estimated CO₂ Emissions (kg) = Sum of (Daily Generation by Fuel Source in MWh × Fuel Emissions Factor in kg CO₂/MWh)**

The `demand_mwh` variable will remain unchanged because it represents daily average hourly electricity consumption rather than total daily consumption.

In [8]:
# Load the complete fuel emissions factor dataset
fuel_factors_clean = pd.read_csv("fuel_factors_clean.csv")

# Verify the fuel emissions factor dataset
expected_fuels = [
    "coal",
    "natural_gas",
    "nuclear",
    "wind",
    "solar",
    "hydro",
    "other",
    "imports"
]

missing_fuels = [
    fuel for fuel in expected_fuels
    if fuel not in fuel_factors_clean["fuel"].values
]

if missing_fuels:
    raise ValueError(
        f"Missing fuel emissions factors: {missing_fuels}"
    )

print("All eight fuel emissions factors are available.")
display(fuel_factors_clean)

# Identify generation sources available in the daily dataset
generation_fuels = [
    "coal",
    "natural_gas",
    "nuclear",
    "wind",
    "solar",
    "hydro",
    "other"
]

# Create a dictionary of emissions factors
emission_factors = (
    fuel_factors_clean
    .set_index("fuel")["kg_co2_per_mwh"]
    .to_dict()
)

# Verify that every generation source has an emissions factor
missing_factors = [
    fuel for fuel in generation_fuels
    if fuel not in emission_factors
]

if missing_factors:
    raise ValueError(
        f"Missing emissions factors for: {missing_factors}"
    )

# Verify that emissions factors are numeric
for fuel in generation_fuels:
    if pd.isna(emission_factors[fuel]):
        raise ValueError(f"Missing emissions factor for {fuel}")

# Convert average hourly generation into estimated daily MWh
# Assumes generation columns contain daily average hourly values.
for fuel in generation_fuels:
    daily_data[f"{fuel}_daily_mwh"] = (
        daily_data[fuel] * daily_data["hours_recorded"]
    )

# Calculate CO2 emissions for each generation source
for fuel in generation_fuels:
    daily_data[f"{fuel}_co2_kg"] = (
        daily_data[f"{fuel}_daily_mwh"]
        * emission_factors[fuel]
    )

# Calculate total estimated daily grid CO2 emissions
emission_columns = [
    f"{fuel}_co2_kg" for fuel in generation_fuels
]

daily_data["daily_co2_kg"] = (
    daily_data[emission_columns].sum(axis=1)
)

# Display the calculated emissions
display(
    daily_data[
        ["date", "hours_recorded", "daily_co2_kg"]
        + emission_columns
    ].head()
)

All eight fuel emissions factors are available.


,fuel,kg_co2_per_mwh,note
0,coal,1010.0,fleet average of the BA's coal units
1,natural_gas,430.0,fleet average of the BA's gas units (combined ...
2,nuclear,0.0,no stack emissions
3,wind,0.0,no stack emissions
4,solar,0.0,no stack emissions
5,hydro,0.0,no stack emissions
6,other,0.0,"landfill gas and biomass; biogenic, counted as..."
7,imports,520.0,average of the neighbouring balancing authorit...


,date,hours_recorded,daily_co2_kg,coal_co2_kg,natural_gas_co2_kg,nuclear_co2_kg,wind_co2_kg,solar_co2_kg,hydro_co2_kg,other_co2_kg
0,2022-01-01,24,8300850.0,7337650.0,963200.0,0.0,0.0,0.0,0.0,0.0
1,2022-01-02,24,9068220.0,8092120.0,976100.0,0.0,0.0,0.0,0.0,0.0
2,2022-01-03,24,11312930.0,9607120.0,1705810.0,0.0,0.0,0.0,0.0,0.0
3,2022-01-04,24,9419790.0,8305230.0,1114560.0,0.0,0.0,0.0,0.0,0.0
4,2022-01-05,24,9593550.0,8528440.0,1065110.0,0.0,0.0,0.0,0.0,0.0


### 3.3 Verify the Generation Totals

The seven available generation categories will be added together and compared with the reported `net_generation` value.

This check helps determine whether the listed generation categories account for the reported generation total.

The combined dataset does not contain a separate imports generation column. Therefore, import-related emissions cannot be calculated independently using the imports emissions factor.

The estimated emissions calculation will use the seven available generation categories. The results should be interpreted with this limitation in mind.

In [9]:
# Calculate the sum of the available generation sources
daily_data["generation_sum"] = daily_data[
    generation_fuels
].sum(axis=1)

# Compare the generation sum with reported net generation
daily_data["generation_difference"] = (
    daily_data["net_generation"]
    - daily_data["generation_sum"]
)

# Display the maximum difference
print(
    "Maximum absolute generation difference:",
    daily_data["generation_difference"].abs().max()
)

# Display a sample of the comparison
display(
    daily_data[
        [
            "date",
            "net_generation",
            "generation_sum",
            "generation_difference"
        ]
    ].head()
)

Maximum absolute generation difference: 2.2737367544323206e-13


,date,net_generation,generation_sum,generation_difference
0,2022-01-01,743.625000,743.625000,0.000000e+00
1,2022-01-02,741.625000,741.625000,-1.136868e-13
2,2022-01-03,862.958333,862.958333,1.136868e-13
3,2022-01-04,823.416667,823.416667,0.000000e+00
4,2022-01-05,791.500000,791.500000,0.000000e+00


### 3.4 Create Calendar Features

Calendar features will be created from the daily date column to account for recurring patterns in electricity demand and grid emissions.

The following variables will be created:

- **Day of week:** Identifies differences between weekdays and weekends.
- **Day of year:** Represents the seasonal position of each observation.
- **Month:** Identifies monthly patterns.
- **Year:** Distinguishes observations from 2022 and 2023.

These variables will provide additional context for the candidate prediction models when evaluating whether temperature contributes predictive information beyond calendar patterns.

AI usage: Looking up Pandas warnings.

In [11]:
# Ensure the date column uses datetime format
daily_data["date"] = pd.to_datetime(daily_data["date"])

# Create calendar features
daily_data["day_of_week"] = daily_data["date"].dt.day_of_week
daily_data["day_of_year"] = daily_data["date"].dt.day_of_year
daily_data["month"] = daily_data["date"].dt.month
daily_data["year"] = daily_data["date"].dt.year

# Display the new calendar features
display(
    daily_data[
        [
            "date",
            "temp_f",
            "day_of_week",
            "day_of_year",
            "month",
            "year"
        ]
    ].head()
)

,date,temp_f,day_of_week,day_of_year,month,year
0,2022-01-01,43.691667,5,1,1,2022
1,2022-01-02,43.258333,6,2,1,2022
2,2022-01-03,38.054167,0,3,1,2022
3,2022-01-04,42.395833,1,4,1,2022
4,2022-01-05,44.670833,2,5,1,2022


### 3.5 Prepare the Final Modeling Dataset

The final modeling dataset will contain the variables needed to evaluate Hypothesis 2.

These variables include daily average outdoor temperature, daily average electricity consumption, estimated daily grid CO₂ emissions, and calendar features.

Rows with missing values in the required variables will be removed before the experiments and modeling process.

The resulting dataset will be sorted chronologically to support the time-based training, validation, and testing strategy.

In [12]:
# Select the variables needed for Hypothesis 2
model_columns = [
    "date",
    "temp_f",
    "demand_mwh",
    "daily_co2_kg",
    "day_of_week",
    "day_of_year",
    "month",
    "year"
]

# Create the modeling dataset
model_data = daily_data[model_columns].copy()

# Remove rows with missing required values
model_data = model_data.dropna()

# Sort observations chronologically
model_data = (
    model_data
    .sort_values("date")
    .reset_index(drop=True)
)

# Display dataset information
print("Final modeling dataset shape:", model_data.shape)

print(
    "Date range:",
    model_data["date"].min(),
    "to",
    model_data["date"].max()
)

print("\nMissing values:")
display(model_data.isnull().sum())

display(model_data.head())

Final modeling dataset shape: (730, 8)
Date range: 2022-01-01 00:00:00 to 2023-12-31 00:00:00

Missing values:


date            0
temp_f          0
demand_mwh      0
daily_co2_kg    0
day_of_week     0
day_of_year     0
month           0
year            0
dtype: int64

,date,temp_f,demand_mwh,daily_co2_kg,day_of_week,day_of_year,month,year
0,2022-01-01,43.691667,743.625000,8300850.0,5,1,1,2022
1,2022-01-02,43.258333,741.583333,9068220.0,6,2,1,2022
2,2022-01-03,38.054167,863.083333,11312930.0,0,3,1,2022
3,2022-01-04,42.395833,823.166667,9419790.0,1,4,1,2022
4,2022-01-05,44.670833,791.458333,9593550.0,2,5,1,2022


# 4. R6 — Cold-versus-Warm Experiments

The purpose of these experiments is to determine whether colder daily temperatures are associated with higher daily average electricity consumption and estimated grid CO₂ emissions.

Two separate comparisons will be conducted:

1. Daily average electricity consumption on cold days versus warm days.
2. Daily estimated grid CO₂ emissions on cold days versus warm days.

Cold and warm days will be identified using the distribution of daily average outdoor temperatures.

The coldest 25% of observations will represent the cold group, while the warmest 25% will represent the warm group. The middle 50% will be excluded from these comparisons.

Both experiments will use the same temperature groups, observation period, and comparison methods. This will allow the results to be evaluated consistently.

### 4.2 Experiment Design

**Experiment 1 — Electricity Consumption**

Compare the daily average hourly electricity consumption between the coldest and warmest temperature groups.

**Experiment 2 — Estimated Grid CO₂ Emissions**

Compare estimated daily grid CO₂ emissions between the coldest and warmest temperature groups.

**Held Fixed:**

- The same combined daily dataset.
- The same date range.
- The same temperature group definitions.
- The same comparison calculations.
- The same measurement units and definitions.

**Varied:**

- Temperature group: coldest 25% versus warmest 25%.

**Measurements:**

- Number of observations in each group.
- Mean and median electricity consumption.
- Mean and median estimated daily CO₂ emissions.
- Absolute difference between cold and warm groups.
- Percentage difference between cold and warm groups.

The comparisons will identify associations between temperature and the two outcomes. They will not establish that temperature directly caused the observed differences.

### 4.3 Define the Comparison Calculations

The coldest and warmest temperature groups will be compared using their daily average electricity consumption and estimated daily grid CO₂ emissions.

The following calculations will be used for both experiments.

**1. Absolute Difference**

Measures how much higher or lower the cold group's average is compared with the warm group's average.

Absolute Difference = Cold Group Mean − Warm Group Mean

**2. Percentage Difference**

Measures the relative difference between the cold and warm group averages.

Percentage Difference = [(Cold Group Mean − Warm Group Mean) / Warm Group Mean] × 100

**3. Group Sample Size**

Represents the number of daily observations included in each temperature group.

**4. Mean**

Represents the average value of electricity consumption or estimated grid CO₂ emissions within each group.

**5. Median**

Represents the middle value after sorting the observations from lowest to highest. The median is included because it is less sensitive to unusually high or low observations than the mean.

**Interpretation:**

- A positive absolute difference indicates that the cold group has a higher average.
- A negative absolute difference indicates that the cold group has a lower average.
- A percentage difference above zero indicates a higher cold-group average relative to the warm group.
- A percentage difference below zero indicates a lower cold-group average relative to the warm group.

These calculations will help evaluate the two predictions in Hypothesis 2.

### 4.4 Seasonal and Calendar Considerations

Temperature groups may contain different seasonal and calendar patterns. For example, cold days may occur more frequently during winter, while warm days may occur more frequently during summer.

Electricity consumption and grid emissions can also vary depending on the season, day of the week, and other calendar patterns.

The following factors may influence the observed differences:

- **Seasonality:** Differences between cold and warm days may partly reflect seasonal changes rather than temperature alone.
- **Day of the week:** Electricity consumption may differ between weekdays and weekends.
- **Heating demand:** Colder temperatures may increase electricity use for heating, although some heating systems use natural gas instead.
- **Generation mix:** Higher electricity consumption does not necessarily produce higher CO₂ emissions because generation sources have different emissions factors.
- **External influences:** Generation availability and broader grid conditions may also affect daily consumption and emissions.

Calendar features will be included in the candidate prediction models to evaluate whether temperature provides additional predictive information beyond recurring calendar patterns.

The cold-versus-warm comparisons will identify associations rather than establish that colder temperatures directly caused changes in electricity consumption or grid emissions.

### 4.5 Identify Cold and Warm Days

Daily observations will be divided into temperature groups using the 25th and 75th percentiles of daily average outdoor temperature.

- **Cold group:** Days at or below the 25th percentile.
- **Warm group:** Days at or above the 75th percentile.
- **Middle group:** Days between the two thresholds.

The middle 50% of observations will be excluded from the initial cold-versus-warm comparisons.

The same temperature thresholds and group assignments will be used for both electricity consumption and estimated grid CO₂ emissions experiments.

Because observations with temperatures equal to a threshold are included in that group, the resulting group sizes may not be exactly 25% each.

In [13]:
# Calculate temperature quartile thresholds
cold_threshold = model_data["temp_f"].quantile(0.25)
warm_threshold = model_data["temp_f"].quantile(0.75)

print("Cold temperature threshold:", cold_threshold)
print("Warm temperature threshold:", warm_threshold)

# Assign temperature groups
model_data["temperature_group"] = np.select(
    [
        model_data["temp_f"] <= cold_threshold,
        model_data["temp_f"] >= warm_threshold
    ],
    [
        "Cold",
        "Warm"
    ],
    default="Middle"
)

# Display group counts
print("\nTemperature group counts:")

display(
    model_data["temperature_group"]
    .value_counts()
)

Cold temperature threshold: 45.84791666666666
Warm temperature threshold: 74.84375

Temperature group counts:


temperature_group
Middle    364
Cold      183
Warm      183
Name: count, dtype: int64

### 4.6 Compare Cold and Warm Temperature Groups

Two comparisons will be performed using the temperature groups.

**Experiment 1: Daily Average Electricity Consumption**

The mean and median daily average hourly electricity consumption will be compared between the coldest and warmest temperature groups.

**Experiment 2: Daily Estimated Grid CO₂ Emissions**

The mean and median estimated daily grid CO₂ emissions will be compared between the coldest and warmest temperature groups.

For each experiment, the following results will be reported:

- Number of observations in each group.
- Cold-group and warm-group means.
- Cold-group and warm-group medians.
- Absolute difference.
- Percentage difference.

The results will be used to evaluate whether the observed differences support the two predictions in Hypothesis 2.

In [14]:
def compare_temperature_groups(data, outcome):

    cold = data.loc[
        data["temperature_group"] == "Cold", outcome
    ].dropna()

    warm = data.loc[
        data["temperature_group"] == "Warm", outcome
    ].dropna()

    cold_mean = cold.mean()
    warm_mean = warm.mean()

    # Calculate absolute difference
    difference = cold_mean - warm_mean

    # Calculate percentage difference
    percentage_difference = (
        difference / warm_mean * 100
        if warm_mean != 0 else np.nan
    )

    return {
        "Outcome": outcome,
        "Cold Count": len(cold),
        "Warm Count": len(warm),
        "Cold Mean": cold_mean,
        "Warm Mean": warm_mean,
        "Cold Median": cold.median(),
        "Warm Median": warm.median(),
        "Absolute Difference": difference,
        "Percentage Difference": percentage_difference
    }


# Experiment 1: Electricity consumption
demand_results = compare_temperature_groups(
    model_data,
    "demand_mwh"
)

# Experiment 2: Estimated grid CO2 emissions
emissions_results = compare_temperature_groups(
    model_data,
    "daily_co2_kg"
)

# Combine both experiment results
experiment_results = pd.DataFrame([
    demand_results,
    emissions_results
])

display(experiment_results)

,Outcome,Cold Count,Warm Count,Cold Mean,Warm Mean,Cold Median,Warm Median,Absolute Difference,Percentage Difference
0,demand_mwh,183,183,8.514346e+02,7.715503e+02,8.314583e+02,772.5,7.988428e+01,10.353735
1,daily_co2_kg,183,183,1.053044e+07,8.794631e+06,1.056266e+07,8780950.0,1.735807e+06,19.737124


### 4.7 Experiment Results and Interpretation

The `demand_mwh` variable represents the daily average of hourly electricity consumption across the balancing area. This includes consumption from homes, businesses, industry, streetlights, and EV chargers.

The results from both experiments will be evaluated independently.

**Electricity Consumption:**

A positive difference between the cold and warm groups would support the prediction that colder days are associated with higher daily average electricity consumption.

A negative difference would indicate that the cold group had lower average electricity consumption than the warm group.

**Estimated Grid CO₂ Emissions:**

A positive difference would support the prediction that colder days are associated with higher estimated daily grid CO₂ emissions.

A negative difference would indicate that the cold group had lower average estimated emissions than the warm group.

A difference close to zero would indicate that the groups had similar averages.

The results will be interpreted alongside seasonal and calendar limitations. These comparisons describe differences between temperature groups and do not establish that colder temperatures directly caused changes in electricity consumption or emissions.

## 5. Candidate Models and Feature Sets

The purpose of this experiment is to evaluate whether adding daily average outdoor temperature improves the ability to predict daily electricity demand and estimated grid CO2 emissions.

Three candidate feature sets will be compared using the same Linear Regression algorithm. Each candidate adds a specific set of information so that changes in model performance can be attributed to the additional features.

The experiment will evaluate two separate prediction targets:

1. Daily average electricity demand (`demand_mwh`).
2. Estimated daily grid CO2 emissions (`daily_co2_kg`).

All candidate models will use the same chronological training, validation, and test split. The same preprocessing approach and evaluation metrics will also be applied to every candidate.

### 5.2 Candidate Feature Sets

Three candidate feature sets will be evaluated.

**Candidate 1: Calendar Baseline**

Features:
- `day_of_week`
- `day_of_year`

This model establishes baseline predictive performance using calendar information. It accounts for recurring differences associated with weekdays and seasonal timing without including temperature.

**Candidate 2: Calendar + Temperature**

Features:
- `day_of_week`
- `day_of_year`
- `temp_f_centered`

This model adds daily average outdoor temperature to determine whether temperature provides useful predictive information beyond calendar patterns.

**Candidate 3: Calendar + Temperature + Nonlinear Temperature**

Features:
- `day_of_week`
- `day_of_year`
- `temp_f_centered`
- `temp_f_squared`

This model adds a squared temperature feature to allow the relationship between temperature and the prediction target to be nonlinear. For example, electricity demand may increase during both unusually cold and unusually hot conditions.

The three candidates use the same algorithm. The only planned difference between them is the information supplied through their feature sets.

In [16]:
# Define candidate feature sets for Hypothesis 2

feature_sets = {
    "calendar_baseline": [
        "day_of_week",
        "day_of_year"
    ],

    "calendar_temperature": [
        "day_of_week",
        "day_of_year",
        "temp_f_centered"
    ],

    "calendar_temperature_squared": [
        "day_of_week",
        "day_of_year",
        "temp_f_centered",
        "temp_f_squared"
    ]
}

# Display candidate feature sets
for model_name, features in feature_sets.items():
    print(f"\n{model_name}")
    print(f"Features: {features}")


calendar_baseline
Features: ['day_of_week', 'day_of_year']

calendar_temperature
Features: ['day_of_week', 'day_of_year', 'temp_f_centered']

calendar_temperature_squared
Features: ['day_of_week', 'day_of_year', 'temp_f_centered', 'temp_f_squared']


### 5.4 Prediction Targets

Each candidate feature set will be evaluated separately against two prediction targets.

- `demand_mwh`: Daily average electricity demand in megawatt-hours.
- `daily_co2_kg`: Estimated daily grid CO2 emissions in kilograms.

This produces six candidate model configurations: three feature sets for each of the two prediction targets.

The primary comparison will be validation RMSE. MAE and R-squared will also be recorded to provide additional information about prediction error and model fit.

In [ ]:
# Define the two prediction targets
target_variables = {
    "electricity_demand": "demand_mwh",
    "grid_co2_emissions": "daily_co2_kg"
}

print("Prediction targets:")
for name, column in target_variables.items():
    print(f"{name}: {column}")

### 5.5 Experimental Controls

To ensure a fair comparison:

- All candidates will use the same Linear Regression algorithm.
- All candidates will use the same chronological data split.
- All candidates will use the same training, validation, and test observations.
- All candidates will use the same preprocessing approach.
- All candidates will be evaluated using MAE, RMSE, and R-squared.
- Temperature transformations will be calculated using the training data mean only.

The baseline model will be compared against the temperature-based models to determine whether adding temperature improves predictive performance.

## 6. Data Splitting and Validation Strategy

### 6.1 Chronological Data Split

Because this project uses daily observations collected over time, a chronological split will be used instead of a random split.

Random splitting could allow future observations to appear in the training data while earlier observations appear in the validation or testing data. This would not accurately represent how a model would perform when predicting future electricity demand and emissions.

The dataset will be divided into three chronological subsets:

- **Training set (70%):** Used to train the candidate models.
- **Validation set (15%):** Used to compare candidate models and select the best feature set.
- **Testing set (15%):** Reserved for the final evaluation of the selected model.

The observations will remain sorted by date, with earlier dates assigned to training and later dates assigned to validation and testing.

The same split will be reused for all six candidate model runs.

### 6.2 Define the Shared Training, Validation, and Test Sets

The data will be divided once using chronological ordering. The resulting indices will be reused for every candidate model and both prediction targets.

No observations from the validation or testing periods will be used to train the models. The final test set will remain untouched until the candidate models have been compared and the best model has been selected.

In [18]:
# Ensure observations are sorted chronologically
model_data = (
    model_data.sort_values("date")
    .reset_index(drop=True)
)

# Calculate split boundaries
total_rows = len(model_data)

train_end = int(total_rows * 0.70)
validation_end = int(total_rows * 0.85)

# Create chronological subsets
train_data = model_data.iloc[:train_end].copy()

validation_data = model_data.iloc[
    train_end:validation_end
].copy()

test_data = model_data.iloc[
    validation_end:
].copy()

# Display split information
print("Total observations:", total_rows)

print("\nTraining set:")
print("Rows:", len(train_data))
print("Start:", train_data["date"].min())
print("End:", train_data["date"].max())

print("\nValidation set:")
print("Rows:", len(validation_data))
print("Start:", validation_data["date"].min())
print("End:", validation_data["date"].max())

print("\nTesting set:")
print("Rows:", len(test_data))
print("Start:", test_data["date"].min())
print("End:", test_data["date"].max())

Total observations: 730

Training set:
Rows: 510
Start: 2022-01-01 00:00:00
End: 2023-05-25 00:00:00

Validation set:
Rows: 110
Start: 2023-05-26 00:00:00
End: 2023-09-12 00:00:00

Testing set:
Rows: 110
Start: 2023-09-13 00:00:00
End: 2023-12-31 00:00:00


In [19]:
# Calculate the temperature mean using training data only
temperature_mean = train_data["temp_f"].mean()

# Apply the same training mean to all three datasets
for dataset in [train_data, validation_data, test_data]:
    dataset["temp_f_centered"] = (
        dataset["temp_f"] - temperature_mean
    )

    dataset["temp_f_squared"] = (
        dataset["temp_f_centered"] ** 2
    )

print(f"Training temperature mean: {temperature_mean:.2f} °F")

Training temperature mean: 57.11 °F


### 6.3 Verify Chronological Separation

The split will be checked to ensure that the training, validation, and testing periods do not overlap and that dates remain in chronological order.

This verification helps prevent accidental data leakage between the three subsets.

In [20]:
# Verify chronological ordering
assert train_data["date"].max() < validation_data["date"].min()

assert validation_data["date"].max() < test_data["date"].min()

# Verify there are no overlapping dates
assert set(train_data["date"]).isdisjoint(validation_data["date"])
assert set(train_data["date"]).isdisjoint(test_data["date"])
assert set(validation_data["date"]).isdisjoint(test_data["date"])

# Verify all observations are accounted for
assert (
    len(train_data)
    + len(validation_data)
    + len(test_data)
    == len(model_data)
)

print("Chronological split verified successfully.")
print("No overlapping dates or missing observations between subsets.")

Chronological split verified successfully.
No overlapping dates or missing observations between subsets.


### 6.4 Time-Series Cross-Validation

Time-series cross-validation will be performed using only the training dataset.

The `TimeSeriesSplit` method will create three chronological folds. In each fold, the model will train on earlier observations and validate on the next group of observations.

This provides multiple estimates of model performance and helps evaluate whether results are stable across different time periods.

The validation set will still be used for final candidate-model comparison, while the test set will remain reserved for one final evaluation after model selection.

AI Usage: Before running on the VM, I did this locally, so I needed to use AI to help with MLFlow and using it locally. 

In [23]:
from sklearn.model_selection import TimeSeriesSplit

# Define three chronological cross-validation folds
time_series_cv = TimeSeriesSplit(n_splits=3)

# Display fold sizes and date ranges
for fold_number, (train_indices, validation_indices) in enumerate(
    time_series_cv.split(train_data), start=1
):
    fold_train = train_data.iloc[train_indices]
    fold_validation = train_data.iloc[validation_indices]

    print(f"\nFold {fold_number}")

    print("Training observations:", len(fold_train))
    print("Training dates:",
          fold_train["date"].min(),
          "to",
          fold_train["date"].max())

    print("Validation observations:", len(fold_validation))
    print("Validation dates:",
          fold_validation["date"].min(),
          "to",
          fold_validation["date"].max())


Fold 1
Training observations: 129
Training dates: 2022-01-01 00:00:00 to 2022-05-09 00:00:00
Validation observations: 127
Validation dates: 2022-05-10 00:00:00 to 2022-09-13 00:00:00

Fold 2
Training observations: 256
Training dates: 2022-01-01 00:00:00 to 2022-09-13 00:00:00
Validation observations: 127
Validation dates: 2022-09-14 00:00:00 to 2023-01-18 00:00:00

Fold 3
Training observations: 383
Training dates: 2022-01-01 00:00:00 to 2023-01-18 00:00:00
Validation observations: 127
Validation dates: 2023-01-19 00:00:00 to 2023-05-25 00:00:00


### 6.5 Model Evaluation Metrics

The candidate models will be evaluated using three regression metrics:

- **Mean Absolute Error (MAE):** Measures the average absolute difference between predicted and actual values. Lower values indicate better predictions.
- **Root Mean Squared Error (RMSE):** Measures prediction error while penalizing larger errors more heavily. Lower values indicate better predictions.
- **R-squared (R²):** Measures how much variation in the outcome is explained by the model. Values closer to 1 indicate a better fit, although negative values are possible.

Cross-validation results will be summarized using the mean and standard deviation of each metric across folds.

The difference between training and validation performance will also be examined to identify possible overfitting.

All metrics will be calculated separately for electricity demand and estimated grid CO₂ emissions.

The difference between training RMSE and validation RMSE will be used to examine possible overfitting. A substantially higher validation error may indicate that the model fits the training observations better than unseen observations.

Cross-validation RMSE will also be summarized using its mean and standard deviation. A lower mean indicates better average predictive performance, while a lower standard deviation indicates more consistent performance across the chronological folds.

### 6.6 Final Model Evaluation

The validation results and cross-validation performance will be used to select the best candidate feature set for each prediction target.

After model selection, the selected model will be evaluated on the reserved test dataset. This final evaluation will provide an estimate of how well the model performs on unseen future observations.

The test dataset will not be used to select features, tune models, or make other modeling decisions. It will be evaluated only once for each selected final model.

In [2]:
# Connect to the provided remote MLflow server

import os
import requests
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Configure remote tracking
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_SERVER
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_SERVER)

# Confirm server access
reply = requests.get(
    MLFLOW_SERVER.rstrip("/") + "/api/2.0/mlflow/experiments/search",
    params={"max_results": 1},
    auth=(MLFLOW_USERNAME, MLFLOW_PASSWORD),
    timeout=10
)

print("Server response:", reply.status_code)

if reply.status_code == 200:
    EXPERIMENT = f"Group1-Climate-Energy-{MLFLOW_USERNAME}"
    mlflow.set_experiment(EXPERIMENT)

    print("Connected successfully!")
    print("Tracking URI:", mlflow.get_tracking_uri())
    print("Experiment:", EXPERIMENT)

elif reply.status_code == 401:
    raise Exception("Incorrect MLflow username or password.")

elif reply.status_code == 403:
    raise Exception("Your account does not have permission to access MLflow.")

else:
    raise Exception(f"MLflow server returned HTTP {reply.status_code}")

Server response: 200


2026/11/11 14:58:52 INFO mlflow.tracking.fluent: Experiment with name 'Group1-Climate-Energy-csc-4933-001-user01' does not exist. Creating a new experiment.


Connected successfully!
Tracking URI: http://10.0.100.20:5000
Experiment: Group1-Climate-Energy-csc-4933-001-user01


## 7. Model Training and Experiment Tracking

### 7.1 Regression Model

Linear Regression will be used for all candidate models. This algorithm provides a consistent baseline for comparing the predictive value of different feature sets.

A StandardScaler will be included in a pipeline to standardize the input features. The scaler will be fitted only on the training portion of each model run to prevent information from the validation data from leaking into model training.

Each candidate feature set will be evaluated separately for both electricity demand and estimated grid CO₂ emissions, producing six candidate model runs.

The evaluation will include:
- Training MAE, RMSE, and R².
- Validation MAE, RMSE, and R².
- Cross-validation mean and standard deviation for each metric.
- The difference between training and validation RMSE.

MLflow will record the model parameters, feature sets, evaluation metrics, and trained model artifacts for reproducibility.

### 7.1 Variable Selection and Feature Relationships

The candidate feature sets are designed to evaluate whether additional variables improve predictive performance.

Following the bias-variance tradeoff, a simpler model may underfit by ignoring relevant information, while a more complex model may overfit by capturing patterns that do not generalize to unseen observations.

The candidate models will therefore be compared to determine whether adding temperature and temperature squared provides meaningful improvements.

Feature relationships will also be examined to identify possible redundancy and multicollinearity. Temperature and temperature squared are related by construction, so their individual regression coefficients will not be interpreted independently.

The goal is to select a feature set that provides useful predictive information without introducing unnecessary complexity.

In [25]:
# Examine feature relationships using training data only
correlation_features = [
    "day_of_week",
    "day_of_year",
    "temp_f_centered",
    "temp_f_squared"
]

feature_correlations = train_data[
    correlation_features[:2] + ["temp_f"]
].corr()

display(feature_correlations)

# Check correlation between centered temperature and its square
temperature_relationship = train_data[
    ["temp_f_centered", "temp_f_squared"]
].corr()

display(temperature_relationship)

,day_of_week,day_of_year,temp_f
day_of_week,1.000000,0.002108,-0.017828
day_of_year,0.002108,1.000000,0.344065
temp_f,-0.017828,0.344065,1.000000


,temp_f_centered,temp_f_squared
temp_f_centered,1.000000,-0.112213
temp_f_squared,-0.112213,1.000000


### 7.2 Experiment Tracking Setup

MLflow will be used to track each candidate model run. Each run will have a unique run ID and record its prediction target, feature set, model parameters, evaluation metrics, and model artifact.

The experiment results will also be exported to a CSV file for reporting and comparison.

In [3]:
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("MLflow experiment configured.")

MLflow experiment configured.


### 7.3 Evaluation Functions

A shared evaluation function will calculate MAE, RMSE, and R² for each dataset.

The same function will be used for training, validation, and cross-validation results to ensure consistent evaluation across all candidate models.

In [2]:
def calculate_metrics(actual, predicted):
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "r2": r2_score(actual, predicted)
    }

### 7.4 Candidate Model Experiments

Each candidate feature set will be trained using the same Linear Regression pipeline.

For each prediction target, the model will be trained using the training dataset, evaluated using the validation dataset, and evaluated across the three chronological cross-validation folds defined in Section 6.

The validation RMSE will be used as the primary metric for comparing candidate models. Cross-validation results will provide additional information about model stability.

The final testing dataset will not be used during these experiments.

In [ ]:

# Store candidate models and experiment results
candidate_models = {}
candidate_results = []

# Confirm that the remote MLflow experiment is active
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow experiment:", EXPERIMENT)

for target_name, target_column in target_variables.items():

    for feature_name, features in feature_sets.items():

        print(f"\nRunning: {target_name} | {feature_name}")

        X_train = train_data[features]
        y_train = train_data[target_column]

        X_validation = validation_data[features]
        y_validation = validation_data[target_column]

        # Create the same pipeline for every candidate
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("regression", LinearRegression())
        ])

        # Train model
        model.fit(X_train, y_train)

        # Training metrics
        train_predictions = model.predict(X_train)
        train_metrics = calculate_metrics(
            y_train, train_predictions
        )

        # Validation metrics
        validation_predictions = model.predict(X_validation)
        validation_metrics = calculate_metrics(
            y_validation, validation_predictions
        )

        # Chronological cross-validation on training data
        cv_metrics = {
            "mae": [],
            "rmse": [],
            "r2": []
        }

        for train_idx, val_idx in time_series_cv.split(train_data):

            fold_train = train_data.iloc[train_idx]
            fold_validation = train_data.iloc[val_idx]

            fold_model = Pipeline([
                ("scaler", StandardScaler()),
                ("regression", LinearRegression())
            ])

            fold_model.fit(
                fold_train[features],
                fold_train[target_column]
            )

            predictions = fold_model.predict(
                fold_validation[features]
            )

            fold_results = calculate_metrics(
                fold_validation[target_column],
                predictions
            )

            for metric in cv_metrics:
                cv_metrics[metric].append(
                    fold_results[metric]
                )

        # Calculate CV mean and standard deviation
        cv_summary = {}

        for metric, values in cv_metrics.items():
            cv_summary[f"cv_{metric}_mean"] = np.mean(values)
            cv_summary[f"cv_{metric}_std"] = np.std(
                values, ddof=1
            )

        # Training-validation gap
        rmse_gap = (
            validation_metrics["rmse"]
            - train_metrics["rmse"]
        )

        run_name = f"H2_{target_name}_{feature_name}"

        # Log experiment to the REMOTE MLflow server
        with mlflow.start_run(run_name=run_name):

            # Experiment parameters
            mlflow.log_param("hypothesis", "Hypothesis 2")
            mlflow.log_param("algorithm", "LinearRegression")
            mlflow.log_param("target", target_name)
            mlflow.log_param("target_column", target_column)
            mlflow.log_param("feature_set", feature_name)
            mlflow.log_param("features", ", ".join(features))
            mlflow.log_param("scaler", "StandardScaler")
            mlflow.log_param("cv_folds", 3)
            mlflow.log_param("data_split", "chronological_70_15_15")

            # Experiment tags
            mlflow.set_tag("model_stage", "candidate")
            mlflow.set_tag("hypothesis", "H2")
            mlflow.set_tag("experiment_group", EXPERIMENT)

            # Training metrics
            for metric, value in train_metrics.items():
                mlflow.log_metric(
                    f"train_{metric}", float(value)
                )

            # Validation metrics
            for metric, value in validation_metrics.items():
                mlflow.log_metric(
                    f"validation_{metric}", float(value)
                )

            # Cross-validation metrics
            for metric, value in cv_summary.items():
                mlflow.log_metric(metric, float(value))

            # Training-validation gap
            mlflow.log_metric(
                "train_validation_rmse_gap",
                float(rmse_gap)
            )

            # Save experiment configuration
            mlflow.log_dict(
                {
                    "hypothesis": "Hypothesis 2",
                    "target": target_column,
                    "feature_set": feature_name,
                    "features": features,
                    "algorithm": "LinearRegression",
                    "scaler": "StandardScaler",
                    "cv_folds": 3
                },
                "experiment_configuration.json"
            )

            # Save trained model artifact
            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="model"
            )

            # Capture MLflow run ID
            run_id = mlflow.active_run().info.run_id

        # Store model and results locally for later comparison
        candidate_models[(target_name, feature_name)] = model

        result = {
            "run_name": run_name,
            "target": target_name,
            "feature_set": feature_name,
            "run_id": run_id,
            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "train_r2": train_metrics["r2"],
            "validation_mae": validation_metrics["mae"],
            "validation_rmse": validation_metrics["rmse"],
            "validation_r2": validation_metrics["r2"],
            "train_validation_rmse_gap": rmse_gap,
            **cv_summary
        }

        candidate_results.append(result)

        print("MLflow Run ID:", run_id)
        print("Validation RMSE:", validation_metrics["rmse"])
        print("CV RMSE mean:", cv_summary["cv_rmse_mean"])
        print("CV RMSE std:", cv_summary["cv_rmse_std"])

print("\nAll six candidate experiments completed.")
print("Experiments logged to:", mlflow.get_tracking_uri())


NameError: name 'target_variables' is not defined

### 7.5 Candidate Model Results

The results from all six candidate model runs will be combined into one comparison table.

The validation RMSE will be the primary selection metric. Cross-validation RMSE mean and standard deviation will be considered when evaluating model stability.

The training-validation RMSE gap will also be reviewed. A large positive gap may indicate that a model fits the training data substantially better than unseen validation observations.

In [ ]:
# Combine candidate model results
candidate_results_df = pd.DataFrame(candidate_results)

# Display the most important comparison metrics
display(
    candidate_results_df[
        [
            "target",
            "feature_set",
            "validation_mae",
            "validation_rmse",
            "validation_r2",
            "cv_rmse_mean",
            "cv_rmse_std",
            "train_validation_rmse_gap"
        ]
    ].sort_values(
        ["target", "validation_rmse"]
    )
)

In [ ]:
# Verify the remote experiment and logged runs

experiment = mlflow.get_experiment_by_name(EXPERIMENT)

print("Experiment ID:", experiment.experiment_id)
print("Experiment name:", experiment.name)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

display(
    runs[
        [
            "run_id",
            "tags.mlflow.runName",
            "metrics.validation_rmse",
            "metrics.validation_mae",
            "metrics.validation_r2"
        ]
    ]
)

## 8. Model Comparison and Selection

The three candidate feature sets will be compared separately for electricity demand and estimated grid CO₂ emissions.

The primary selection metric will be validation RMSE. The candidate with the lowest validation RMSE will be considered the strongest initial candidate for each prediction target.

Additional considerations include:

- **Cross-validation RMSE mean:** Measures average predictive error across the chronological folds.
- **Cross-validation RMSE standard deviation:** Measures the consistency of predictive performance across folds.
- **Training-validation RMSE gap:** Helps identify possible overfitting.
- **Validation MAE:** Provides another measure of average prediction error.
- **Validation R²:** Provides information about the proportion of variation explained by the model.

A more complex model will not automatically be selected simply because it contains additional features. Improvements in validation and cross-validation performance must justify the added complexity.

The final test dataset will remain unused until both selected models have been identified.

In [ ]:
error_reduction_results = []

for target_name in target_variables:

    target_results = candidate_results_df[
        candidate_results_df["target"] == target_name
    ]

    baseline_rmse = target_results.loc[
        target_results["feature_set"] == "calendar_baseline",
        "validation_rmse"
    ].iloc[0]

    for _, row in target_results.iterrows():

        reduction = (
            (baseline_rmse - row["validation_rmse"])
            / baseline_rmse
        ) * 100

        error_reduction_results.append({
            "target": target_name,
            "feature_set": row["feature_set"],
            "baseline_validation_rmse": baseline_rmse,
            "candidate_validation_rmse": row["validation_rmse"],
            "error_reduction_percent": reduction
        })

error_reduction_df = pd.DataFrame(error_reduction_results)

display(error_reduction_df)

8.3 Select the best feature set for each target

In [ ]:
# Select the lowest validation RMSE within each target
best_candidates = (
    candidate_results_df
    .sort_values(
        ["target", "validation_rmse"],
        ascending=[True, True]
    )
    .groupby("target")
    .first()
    .reset_index()
)

print("Selected candidate models:")

display(
    best_candidates[
        [
            "target",
            "feature_set",
            "validation_rmse",
            "cv_rmse_mean",
            "cv_rmse_std",
            "train_validation_rmse_gap"
        ]
    ]
)

# Store the selected candidate models
selected_models = {}

for _, row in best_candidates.iterrows():

    target_name = row["target"]
    feature_name = row["feature_set"]

    selected_models[target_name] = {
        "feature_set": feature_name,
        "features": feature_sets[feature_name],
        "model": candidate_models[
            (target_name, feature_name)
        ]
    }

### 8.3 MLflow Experiment Records

All candidate model runs will be retrieved from MLflow and exported to a CSV file.

The exported table will include the run name, run ID, model parameters, and recorded evaluation metrics. This provides a reproducible record of the experiments and allows the results to be reviewed outside the notebook.

The MLflow tracking directory will also retain the individual experiment records and model artifacts.

In [ ]:
# Retrieve the MLflow experiment
experiment = mlflow.get_experiment_by_name(
    "Hypothesis_2_Model_Experiments"
)

mlflow_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

# Keep candidate runs
candidate_mlflow_runs = mlflow_runs[
    mlflow_runs["tags.model_stage"] == "candidate"
].copy()

# Rename columns for the exported table
mlflow_run_table = candidate_mlflow_runs.rename(
    columns={
        "tags.mlflow.runName": "run_name",
        "run_id": "run_id"
    }
)

# Export all run parameters and metrics
mlflow_run_table.to_csv(
    "hypothesis_2_mlflow_runs.csv",
    index=False
)

display(mlflow_run_table)

print("MLflow run table exported successfully.")

## 9. Final Evaluation on Unseen Test Data

After selecting the best candidate feature set for each prediction target, the selected models will be retrained using the combined training and validation datasets.

The reserved test dataset will then be used to evaluate each final model.

This approach allows the selected models to learn from all observations available before the test period while preserving the test observations as unseen data.

The final test evaluation will be performed once for each prediction target using MAE, RMSE, and R².

The test results will provide an estimate of how well the selected models generalize to later observations.

No additional model selection or feature changes will be made based on the final test results.

9,2 retrain  and test selected models

In [ ]:
# Combine training and validation data
development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True
)

final_test_results = []
final_models = {}

for target_name, selection in selected_models.items():

    features = selection["features"]
    target_column = target_variables[target_name]

    X_development = development_data[features]
    y_development = development_data[target_column]

    X_test = test_data[features]
    y_test = test_data[target_column]

    # Train final selected model
    final_model = Pipeline([
        ("scaler", StandardScaler()),
        ("regression", LinearRegression())
    ])

    final_model.fit(X_development, y_development)

    # Evaluate once on the reserved test dataset
    test_predictions = final_model.predict(X_test)

    test_metrics = calculate_metrics(
        y_test,
        test_predictions
    )

    final_models[target_name] = final_model

    final_test_results.append({
        "target": target_name,
        "selected_feature_set": selection["feature_set"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "test_r2": test_metrics["r2"]
    })

    # Log final test evaluation
    with mlflow.start_run(
        run_name=f"FINAL_TEST_{target_name}"
    ):

        mlflow.set_tag("model_stage", "final_test")

        mlflow.log_param("algorithm", "LinearRegression")
        mlflow.log_param("target", target_name)
        mlflow.log_param(
            "selected_feature_set",
            selection["feature_set"]
        )

        mlflow.log_param(
            "features",
            ", ".join(features)
        )

        for metric, value in test_metrics.items():
            mlflow.log_metric(
                f"test_{metric}",
                float(value)
            )

        mlflow.sklearn.log_model(
            sk_model=final_model,
            name="final_model"
        )

# Display final test results
final_test_results_df = pd.DataFrame(
    final_test_results
)

display(final_test_results_df)

compare validation and testresults

### 9.3 Final Test Results

The final test results will be compared with the validation and cross-validation results.

The following measurements will be examined:

- Final test MAE.
- Final test RMSE.
- Final test R².
- Difference between validation RMSE and test RMSE.
- Whether the final test performance is consistent with the earlier validation results.

A substantial increase in test error may indicate that the selected model does not generalize well to later observations.

The test results will be reported separately for electricity demand and estimated grid CO₂ emissions.

These metrics evaluate predictive performance and do not independently establish a causal relationship between temperature and either outcome.

In [ ]:
# Compare validation and final test performance
validation_test_comparison = best_candidates[
    ["target", "feature_set", "validation_rmse"]
].merge(
    final_test_results_df[
        ["target", "test_rmse", "test_mae", "test_r2"]
    ],
    on="target"
)

validation_test_comparison["test_validation_rmse_difference"] = (
    validation_test_comparison["test_rmse"]
    - validation_test_comparison["validation_rmse"]
)

display(validation_test_comparison)

9.4 export final test results

In [ ]:
final_test_results_df.to_csv(
    "hypothesis_2_final_test_results.csv",
    index=False
)

validation_test_comparison.to_csv(
    "hypothesis_2_validation_test_comparison.csv",
    index=False
)

print("Final test results exported.")

### 10.1 Statistical Uncertainty in Temperature Comparisons

The cold-versus-warm comparisons provide the observed differences in daily electricity demand and estimated grid CO₂ emissions.

A bootstrap procedure will be used to estimate 95% confidence intervals around the differences in group means.

The procedure repeatedly samples observations with replacement from the cold and warm groups and calculates the difference between their means.

If the confidence interval is entirely above zero, the cold group has a higher mean. If it is entirely below zero, the cold group has a lower mean. If the interval includes zero, the difference is not clearly distinguishable from zero using this procedure.

Because daily observations may be correlated across nearby dates, the independent bootstrap procedure is an approximation and may underestimate uncertainty. The confidence intervals will therefore be interpreted cautiously.

These comparisons measure associations and do not establish that temperature directly caused the observed differences.

In [ ]:
# Bootstrap confidence interval for cold-minus-warm mean difference

def bootstrap_mean_difference(
    cold_values,
    warm_values,
    n_bootstrap=2000,
    seed=42
):
    cold_values = np.asarray(cold_values)
    warm_values = np.asarray(warm_values)

    rng = np.random.default_rng(seed)

    observed_difference = (
        np.mean(cold_values) - np.mean(warm_values)
    )

    bootstrap_differences = []

    for _ in range(n_bootstrap):
        cold_sample = rng.choice(
            cold_values,
            size=len(cold_values),
            replace=True
        )

        warm_sample = rng.choice(
            warm_values,
            size=len(warm_values),
            replace=True
        )

        difference = (
            np.mean(cold_sample) - np.mean(warm_sample)
        )

        bootstrap_differences.append(difference)

    lower_bound, upper_bound = np.percentile(
        bootstrap_differences,
        [2.5, 97.5]
    )

    return {
        "Observed Difference": observed_difference,
        "95% CI Lower": lower_bound,
        "95% CI Upper": upper_bound
    }


# Evaluate both outcomes
bootstrap_results = []

for outcome in ["demand_mwh", "daily_co2_kg"]:

    cold_values = model_data.loc[
        model_data["temperature_group"] == "Cold",
        outcome
    ].dropna()

    warm_values = model_data.loc[
        model_data["temperature_group"] == "Warm",
        outcome
    ].dropna()

    result = bootstrap_mean_difference(
        cold_values,
        warm_values
    )

    result["Outcome"] = outcome
    bootstrap_results.append(result)

bootstrap_results_df = pd.DataFrame(
    bootstrap_results
)

display(bootstrap_results_df)

### 10.2 Hypothesis Verdict

The corrected Hypothesis 2 predicts that colder daily average outdoor temperatures will be associated with higher daily average electricity demand and higher estimated daily grid CO₂ emissions.

The two predictions will be evaluated independently using the cold-versus-warm comparisons and their bootstrap confidence intervals.

**Possible Verdicts:**

- **Supported:** Both electricity demand and estimated grid CO₂ emissions have positive cold-minus-warm differences with confidence intervals entirely above zero.
- **Partially Supported:** Only one outcome has a positive difference with a confidence interval entirely above zero.
- **Rejected:** Both outcomes have differences in the opposite direction, with confidence intervals entirely below zero.
- **Unresolved:** The results do not provide sufficiently clear evidence to support or reject the predictions.

Predictive model results will provide additional context about whether temperature contributes useful information beyond calendar patterns.

Model performance alone will not determine the hypothesis verdict because predictive relationships do not establish causation.

The final verdict will be based on the observed comparisons, uncertainty intervals, and consistency of the supporting evidence.

In [ ]:
# Classify the direction of each observed difference

def classify_difference(lower, upper):
    if lower > 0:
        return "Cold group significantly higher"
    elif upper < 0:
        return "Cold group significantly lower"
    else:
        return "No clear difference"


bootstrap_results_df["Interpretation"] = (
    bootstrap_results_df.apply(
        lambda row: classify_difference(
            row["95% CI Lower"],
            row["95% CI Upper"]
        ),
        axis=1
    )
)

display(bootstrap_results_df)

# Determine overall hypothesis verdict
demand_result = bootstrap_results_df[
    bootstrap_results_df["Outcome"] == "demand_mwh"
].iloc[0]

emissions_result = bootstrap_results_df[
    bootstrap_results_df["Outcome"] == "daily_co2_kg"
].iloc[0]

demand_higher = demand_result["95% CI Lower"] > 0
demand_lower = demand_result["95% CI Upper"] < 0

emissions_higher = emissions_result["95% CI Lower"] > 0
emissions_lower = emissions_result["95% CI Upper"] < 0

if demand_higher and emissions_higher:
    overall_verdict = "Supported"

elif demand_lower and emissions_lower:
    overall_verdict = "Rejected"

elif demand_higher or emissions_higher:
    overall_verdict = "Partially supported"

elif demand_lower or emissions_lower:
    overall_verdict = "Partially supported (mixed or opposite evidence)"

else:
    overall_verdict = "Unresolved"

print("Overall Hypothesis 2 Verdict:", overall_verdict)

### 10.3 Predictive Model Interpretation

The selected candidate models will be reviewed using their validation, cross-validation, and final test results.

The following questions will guide the interpretation:

1. Did adding temperature improve validation RMSE compared with the calendar baseline?
2. Did adding temperature squared provide additional improvement?
3. Were the cross-validation results reasonably consistent across folds?
4. Did the selected models maintain similar performance on the final test dataset?
5. Did temperature contribute useful predictive information for both outcomes or only one?

The results will help determine whether temperature is useful for predicting daily electricity consumption and estimated grid emissions beyond recurring calendar patterns.

The predictive results will be interpreted separately from the cold-versus-warm comparisons.

If adding temperature reduces validation RMSE compared with the calendar baseline, this suggests temperature provides additional predictive information.

If adding temperature squared produces a further reduction, this suggests that a nonlinear temperature feature may improve predictions.

However, if the more complex models produce little improvement or show less stable cross-validation results, the simpler model may be preferable.

The final test results will be used to determine whether the selected feature sets maintain their predictive performance on later, unseen observations.

These results will help distinguish between observed temperature associations and the ability of temperature to improve predictive accuracy.

### 10.4 Summary of Hypothesis 2 Evidence

The following results will be used to summarize the evidence for Hypothesis 2:

**Cold-versus-Warm Experiments**
- Electricity demand: Cold-group mean, warm-group mean, absolute difference, percentage difference, and 95% confidence interval.
- Grid CO₂ emissions: Cold-group mean, warm-group mean, absolute difference, percentage difference, and 95% confidence interval.

**Model Selection**
- Best electricity demand feature set.
- Best grid CO₂ emissions feature set.
- Validation RMSE for each selected model.
- Cross-validation RMSE mean and standard deviation.
- Training-validation RMSE gap.

**Final Validation**
- Final test MAE, RMSE, and R² for both outcomes.
- Comparison of validation and test performance.

Together, these results will provide the numerical evidence used to determine the final hypothesis verdict.

In [ ]:
# Combine cold-versus-warm and bootstrap results
hypothesis_evidence = experiment_results.merge(
    bootstrap_results_df,
    left_on="Outcome",
    right_on="Outcome",
    how="left"
)

display(hypothesis_evidence)

# Export hypothesis evidence
hypothesis_evidence.to_csv(
    "hypothesis_2_evidence_summary.csv",
    index=False
)

In [ ]:
# Combine selected candidate results with final test results
model_comparison = best_candidates.merge(
    final_test_results_df,
    on="target",
    how="left"
)

display(
    model_comparison[
        [
            "target",
            "feature_set",
            "validation_rmse",
            "cv_rmse_mean",
            "cv_rmse_std",
            "test_rmse",
            "test_mae",
            "test_r2"
        ]
    ]
)

### 10.4 Export Final Results

The final candidate-model comparison, selected-model test results, and bootstrap comparisons will be exported as CSV files.

These files provide supporting numerical evidence for the final project report and preserve the experiment results outside the notebook.

In [ ]:
# Export candidate model results
candidate_results_df.to_csv(
    "hypothesis_2_candidate_results.csv",
    index=False
)

# Export selected model results
model_comparison.to_csv(
    "hypothesis_2_final_model_comparison.csv",
    index=False
)

# Export bootstrap results
bootstrap_results_df.to_csv(
    "hypothesis_2_bootstrap_results.csv",
    index=False
)

print("Final experiment results exported successfully.")

## 11. Limitations and Conclusion

### 11.1 Study Limitations

Several limitations should be considered when interpreting the results.

**Geographic Coverage:** The electricity grid data represents the broader MCPA balancing authority rather than only the specific city where the 10,000 EVs were introduced. Therefore, the analysis cannot attribute all observed changes in electricity demand or emissions directly to the EVs.

**Daily Aggregation:** The analysis uses daily averages rather than hourly observations. This limits the ability to identify specific charging periods, nighttime demand patterns, and short-term changes in electricity consumption.

**Multiple Sources of Electricity Demand:** The electricity demand data includes homes, businesses, industry, streetlights, and EV chargers. The individual contribution of EV charging cannot be separated using the available data.

**Estimated Grid Emissions:** Estimated CO₂ emissions depend on the available generation data and assigned fuel emissions factors. The combined dataset does not provide a separate imports generation column, limiting the ability to calculate imported electricity emissions independently.

**Seasonal and External Influences:** Temperature is not the only factor affecting electricity demand and emissions. Day of the week, seasonal changes, heating requirements, economic activity, and generation availability may also influence the results.

**Limited Observation Period:** The dataset covers 2022 and 2023. The findings may not represent longer-term electricity consumption or generation patterns.

**Predictive Model Limitations:** Linear Regression may not capture every complex relationship between temperature, electricity demand, and grid emissions. Model performance is also limited by the available features and observation period.

Because of these limitations, the findings should be interpreted as observed associations and predictive relationships rather than proof of direct causation.

### 11.2 Hypothesis 2 Conclusion

This experiment examined whether lower daily average outdoor temperatures were associated with higher daily average electricity demand and higher estimated daily grid CO₂ emissions.

The coldest and warmest temperature groups were compared using daily observations from 2022 and 2023. Three candidate feature sets were evaluated using Linear Regression to determine whether temperature and temperature-squared features improved predictions beyond calendar information.

Model selection was based on validation performance and chronological cross-validation. The selected models were then evaluated on the reserved test dataset.

**Final Findings:**

- **Electricity Demand:** [Insert cold-versus-warm difference, confidence interval, and selected model results.]
- **Estimated Grid CO₂ Emissions:** [Insert cold-versus-warm difference, confidence interval, and selected model results.]
- **Best Electricity Demand Model:** [Insert selected feature set.]
- **Best Grid CO₂ Emissions Model:** [Insert selected feature set.]
- **Final Test Performance:** [Insert test RMSE and R² for both outcomes.]
- **Overall Hypothesis Verdict:** [Insert supported, partially supported, rejected, or unresolved.]

The findings will indicate whether colder daily temperatures were associated with higher electricity demand and estimated grid CO₂ emissions during the study period.

The conclusions will remain limited to the available balancing-area data and the two-year observation period.